# Web Scraping

Once the package is installed, we can import it using:

In [2]:
import copy
import pandas as pd
import bs4

from collections import defaultdict
import urllib

Firstly, we will download the HTML source code for our target web page at the link below. This web page contains personal details for a number of customers:

In [3]:
def crawl_and_parse_html(url: str) -> bs4.BeautifulSoup:
    # attempt to open the specified URL
    response = urllib.request.urlopen(url)
    # read the response data (bytes) and decode it into a string
    html = response.read().decode("utf-8")
    # display the HTML string

    return bs4.BeautifulSoup(html, "html.parser")


In [4]:
base_url = "http://mlg.ucd.ie/modules/python/sources/retail"

In [5]:
import urllib.request
import urllib.error

# Index page
index_soup = crawl_and_parse_html(base_url + "/index.html")
print(str(index_soup))

<!DOCTYPE html>

<html lang="en">
<head>
<meta charset="utf-8"/>
<meta content="noindex" name="robots"/>
<meta content="Academic teaching purposes only" name="purpose"/>
<meta content="Retail sales transaction database for educational purposes only" name="description"/>
<meta content="IE=edge" http-equiv="X-UA-Compatible"/>
<meta content="width=device-width, initial-scale=1" name="viewport"/>
<title>Retail Sales Database</title>
<link href="https://maxcdn.bootstrapcdn.com/bootstrap/3.3.5/css/bootstrap.min.css" rel="stylesheet"/>
<script src="https://ajax.googleapis.com/ajax/libs/jquery/1.11.3/jquery.min.js"></script>
<script src="https://maxcdn.bootstrapcdn.com/bootstrap/3.3.5/js/bootstrap.min.js"></script>
<link href="style.css" rel="stylesheet" type="text/css"/>
</head>
<body>
<div class="container mtb">
<header>
<h1>Retail Sales Database</h1>
</header>
<section class="instructions">
<div class="row">
<div class="col-md-12">
<h2>Transaction Data Overview</h2>
<p>
                    

In [6]:
links = index_soup.select("div.quarter-item > h3 > a")
hrefs = [a["href"] for a in links]

hrefs

['2025-Q1-page01.html',
 '2025-Q2-page01.html',
 '2025-Q3-page01.html',
 '2025-Q4-page01.html']

In [7]:
# Concat base url with href for full url of each dataset
quarter_index_urls = [base_url + "/" + href for href in hrefs]
quarter_index_urls

['http://mlg.ucd.ie/modules/python/sources/retail/2025-Q1-page01.html',
 'http://mlg.ucd.ie/modules/python/sources/retail/2025-Q2-page01.html',
 'http://mlg.ucd.ie/modules/python/sources/retail/2025-Q3-page01.html',
 'http://mlg.ucd.ie/modules/python/sources/retail/2025-Q4-page01.html']

In [8]:
# Each quarter index refer to the first page of each quarter, we need to retrieve  all pages of that quarter
item_page_urls = copy.deepcopy(quarter_index_urls)
for url in quarter_index_urls:
    first_page_soup = crawl_and_parse_html(url)
    # print(str(first_page_soup))
    pages = first_page_soup.select("nav.pagination-nav > a")
    # print([a["href"] for a in pages[:-1]])

    item_page_urls.extend([base_url + "/" + a["href"] for a in pages[:-1]])

item_page_urls

['http://mlg.ucd.ie/modules/python/sources/retail/2025-Q1-page01.html',
 'http://mlg.ucd.ie/modules/python/sources/retail/2025-Q2-page01.html',
 'http://mlg.ucd.ie/modules/python/sources/retail/2025-Q3-page01.html',
 'http://mlg.ucd.ie/modules/python/sources/retail/2025-Q4-page01.html',
 'http://mlg.ucd.ie/modules/python/sources/retail/2025-Q1-page02.html',
 'http://mlg.ucd.ie/modules/python/sources/retail/2025-Q1-page03.html',
 'http://mlg.ucd.ie/modules/python/sources/retail/2025-Q1-page04.html',
 'http://mlg.ucd.ie/modules/python/sources/retail/2025-Q1-page05.html',
 'http://mlg.ucd.ie/modules/python/sources/retail/2025-Q1-page06.html',
 'http://mlg.ucd.ie/modules/python/sources/retail/2025-Q1-page07.html',
 'http://mlg.ucd.ie/modules/python/sources/retail/2025-Q1-page08.html',
 'http://mlg.ucd.ie/modules/python/sources/retail/2025-Q1-page09.html',
 'http://mlg.ucd.ie/modules/python/sources/retail/2025-Q1-page10.html',
 'http://mlg.ucd.ie/modules/python/sources/retail/2025-Q1-page11

Print the data that we extracted from the HTML:

In [26]:
data_dict = defaultdict(list)

for url in item_page_urls:
    data_page = crawl_and_parse_html(url)
    items = data_page.select("ol.sales-list li table.transaction-info tr")
    for item in items:
        cols = item.find_all("td")

        header, value = cols[0].get_text(strip=True), cols[1].get_text(strip=True)
        data_dict[header].append(value)
        data_dict[header].append(value)

df = pd.DataFrame(data_dict)

df.to_csv("crawled_data.csv", index=False)

df.head(5)

,Product:,Sale Date:,Category:,Quantity:,Total Price:,Total Profit:,Payment Type:,Customer Details:
0,White Tie and Shirt Set,01-01-2025,Men — Shirts,5,175.00,66.50,Credit Card,Cust ID: 7097From: DublinGender: FemaleAge Ran...
1,White Tie and Shirt Set,01-01-2025,Men — Shirts,5,175.00,66.50,Credit Card,Cust ID: 7097From: DublinGender: FemaleAge Ran...
2,Floral Embroidered Kaftan,2025-01-01,Children — Girls Clothing,1,32.00,16.32,Debit Card,Customer ID: 10753Location LaoisGender: FAge G...
3,Floral Embroidered Kaftan,2025-01-01,Children — Girls Clothing,1,32.00,16.32,Debit Card,Customer ID: 10753Location LaoisGender: FAge G...
4,Black Loose Fit Jeans,2025-01-02,Men — Jeans,1,29.00,11.60,Credit Card,ID 17356Age Category: 55—64City: MayoGender: Male
